# Multi-task Reward Router Evaluation

This notebook evaluates the trained Multi-task Reward Router model. It covers:
1. **Task Classification Accuracy**: How well does the router predict the task type?
2. **Routing Efficiency**: Accuracy, Cost, and Latency analysis comparing the Router's choice vs. Oracle vs. Random.
3. **Failure Analysis**: Investigating cases where the router fails to pick the best model, analyzing confidence scores and thresholds.
4. **Visualizations**: Confusion matrices, performance distributions, and reliability diagrams.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import logging
import json
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModel


# Set project root
current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from artemis_final.router_train.models.reward_router import RewardRouterModel
from artemis_final.router_train.config import RouterModelConfig

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("router_eval")

print(f"Project root set to: {PROJECT_ROOT}")
print(f"Device: {torch.cuda.is_available()'mps' and 'cuda' or 'cpu'}")

# --- NEW: CascadeFlow Setup ---
# Add code_base to path to import CascadeFlow modules
sys.path.append(str(PROJECT_ROOT / "code_base"))
sys.path.append(str(PROJECT_ROOT / "code_base" / "cascadeflow"))

try:
    import nest_asyncio
    nest_asyncio.apply() # Required for async in Jupyter
    
    from cascadeflow import CascadeAgent, ModelConfig
    from cascadeflow.evaluation import Scorer
except ImportError:
    # Fallback if libraries are missing (allows dry run)
    print("Warning: CascadeFlow modules not found. Ensure code_base is in path.")
    CascadeAgent = None
    Scorer = None


Project root set to: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router
Device: cpu


## 1. Data Loading and Preprocessing

In [ ]:
def get_preferred_device():
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        print("Using Apple Silicon MPS (Metal Performance Shaders) GPU")
    else:
        device = torch.device('cpu')
        print("Using CPU")
    return device

class Config:
    # Paths
    data_path = Path(PROJECT_ROOT / "dataset" / "data" /     "router_profiles_with_utility.parquet")
    ckpt_path = Path(PROJECT_ROOT / "artemis_final" / "checkpoints" / "best_multitask_router_v1.pt")
    model_index_path = Path(PROJECT_ROOT / "data" / "model_index.json")
    mode_index_path = Path(PROJECT_ROOT / "data" / "mode_index.json")
    task_index_path = Path(PROJECT_ROOT / "data" / "task_index.json")

cfg = Config()

def load_indices():
    # Try loading saved indices, else we might need to recreate them from data
    try:
        with open(cfg.model_index_path, "r") as f: model_names = json.load(f)
        with open(cfg.mode_index_path, "r") as f: mode_names = json.load(f)
        with open(cfg.task_index_path, "r") as f: task_names = json.load(f)
        print("Loaded indices from disk.")
        return model_names, mode_names, task_names
    except FileNotFoundError:
        print("Indices not found on disk. They will be regenerated from data.")
        return None, None, None

model_names, mode_names, task_names = load_indices()

# Load Data
print(f"Loading data from {cfg.data_path}...")
df = pd.read_parquet(cfg.data_path)
df = df[df["ok"] == True].copy()
print(f"Loaded {len(df)} rows where ok=True")

Loaded indices from disk.
Loading data from /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/examples/data/router_profiles_with_utility.parquet...
Loaded 339056 rows where ok=True


In [3]:
MODES = ["accuracy", "cheap", "fast", "balanced"]

def create_long_format(df, modes):
    rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building long-format"):
        base = {
            "sample_id": row["sample_id"],
            "router_task": row["router_task"],
            "data_split": row["data_split"],
            "prompt_text": row["prompt_text"],
            "model_name": row["model_name"],
            "prompt_len_words": row.get("txt_prompt_length_words", 0),
            "source_dataset": row.get("source_dataset", "unknown"),
            # Metrics for eval
            "actual_score": row.get("utility_accuracy", 0.0), # Assuming this is raw score normalized
            "actual_cost": row.get("estimated_cost_usd", 0.0),
            "actual_latency": row.get("latency_ms", 0.0),
        }
        
        # Add mode rows
        for mode in modes:
            col_name = f"utility_{mode}"
            if col_name in row and pd.notnull(row[col_name]):
                r = base.copy()
                r["mode_name"] = mode
                r["utility_target"] = row[col_name]
                rows.append(r)
    
    return pd.DataFrame(rows)

df_long = create_long_format(df, MODES)

# Regenerate indices if needed
if model_names is None:
    model_names = sorted(df_long["model_name"].unique())
    mode_names = MODES
    task_names = sorted(df_long["router_task"].unique())

model_to_id = {m: i for i, m in enumerate(model_names)}
mode_to_id = {m: i for i, m in enumerate(mode_names)}
task_to_id = {t: i for i, t in enumerate(task_names)}

df_long["model_id"] = df_long["model_name"].map(model_to_id)
df_long["mode_id"] = df_long["mode_name"].map(mode_to_id)
df_long["task_id"] = df_long["router_task"].map(task_to_id)

print(f"Processed Data Shape: {df_long.shape}")
print(f"Models ({len(model_names)}): {model_names}")
print(f"Tasks ({len(task_names)})")

Building long-format:   0%|          | 0/339056 [00:00<?, ?it/s]

Processed Data Shape: (1356224, 15)
Models (5): ['deepseek_ocr', 'gemma_3_27b', 'qwen2_5_vl_3b', 'qwen2_5_vl_7b', 'qwen3_vl_8b_thinking']
Tasks (30)


## 2. Dataset and Model Definition
Defining `MultiTaskRewardRouterDataset` locally to ensure it matches the training logic exactly.

In [4]:
class MultiTaskRewardRouterDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        input_text = (
            f"[ROUTER] Task: {row['router_task']}. "
            f"Source: {row['source_dataset']}. "
            f"PromptLen: {row['prompt_len_words']}. "
            f"Question: {row['prompt_text']}"
        )
        
        encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            truncation=True,
            padding=False,
            return_tensors=None
        )
        
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "model_id": row["model_id"],
            "mode_id": row["mode_id"],
            "utility_target": float(row["utility_target"]),
            "task_id": row["task_id"],
            "sample_id": row["sample_id"]
        }

def multitask_collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    
    max_len = max(len(x) for x in input_ids)
    input_ids_padded = []
    attention_mask_padded = []
    
    for ids, mask in zip(input_ids, attention_mask):
        pad_len = max_len - len(ids)
        input_ids_padded.append(ids + [0] * pad_len)
        attention_mask_padded.append(mask + [0] * pad_len)
        
    return {
        "input_ids": torch.tensor(input_ids_padded, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask_padded, dtype=torch.long),
        "model_id": torch.tensor([x["model_id"] for x in batch], dtype=torch.long),
        "mode_id": torch.tensor([x["mode_id"] for x in batch], dtype=torch.long),
        "utility_target": torch.tensor([x["utility_target"] for x in batch], dtype=torch.float),
        "task_id": torch.tensor([x["task_id"] for x in batch], dtype=torch.long),
        "sample_ids": [x["sample_id"] for x in batch]
    }

In [5]:
# Filter Test Split
test_df = df_long[df_long["data_split"] == "test"]
print(f"Test set size: {len(test_df)}")

# Tokenizer & Dataloader
tokenizer = AutoTokenizer.from_pretrained(cfg.text_encoder_name)
test_ds = MultiTaskRewardRouterDataset(test_df, tokenizer, cfg.max_seq_len)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=multitask_collate_fn)

Test set size: 199860


In [7]:
# Load Model
model_config = RouterModelConfig(
    text_encoder_name=cfg.text_encoder_name,
    model_emb_dim=32,
    mode_emb_dim=16,
    hidden_dim=256,
    dropout=0.1
)

# Re-instantiate the model structure
# Note: The RewardRouterModel in artemis_final.router_train.models.reward_router
# matches the structure used in training. 
model = RewardRouterModel(
    config=model_config,
    num_models=len(model_names),
    num_modes=len(mode_names),
    num_tasks=len(task_names)
)

# Load weights
print(f"Loading checkpoint from {cfg.ckpt_path}...")
state_dict = torch.load(cfg.ckpt_path, map_location=cfg.device)
# If state dict was saved differently, adjust keys
if "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]

model.load_state_dict(state_dict)
model.to(cfg.device)
model.eval()
print("Model loaded successfully.")

INFO:artemis_final.router_train.models.reward_router:Loading text encoder: distilbert-base-uncased
INFO:artemis_final.router_train.models.reward_router:Freezing text encoder parameters
INFO:artemis_final.router_train.models.reward_router:Model initialized with 446,975 trainable parameters


Loading checkpoint from /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/checkpoints/best_multitask_router_v1.pt...
Model loaded successfully.


## 3. Evaluation Loop
Running inference on the test set to collect predictions.

In [ ]:
eval_records = []
task_preds_all = []
task_targets_all = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test Inference"):
        batch = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            model_id=batch["model_id"],
            mode_id=batch["mode_id"],
        )
        
        utility_hat = outputs["utility_hat"]
        task_logits = outputs["task_logits"]
        task_preds = task_logits.argmax(dim=-1)
        
        task_preds_all.extend(task_preds.cpu().tolist())
        task_targets_all.extend(batch["task_id"].cpu().tolist())
        
        # Retrieve metadata from original DF for this batch to get cost/latency/score
        # The batch order is preserved from test_df
        # But simpler is to reconstruct the index or lookup by sample_id + model_id + mode_id if needed
        # Here we just iterate batch items and append results
        
        for i in range(len(batch["sample_ids"])):
            sample_id = batch["sample_ids"][i]
            model_id_int = int(batch["model_id"][i].cpu())
            mode_id_int = int(batch["mode_id"][i].cpu())
            
            # Hacky lookup to get original metrics (since they aren't in the dataset item)
            # Creating a lookup dict beforehand would be faster but let's see if this works efficiently
            # Actually, `test_df` has them. We need to sync indices.
            # `test_ds` uses `test_df.iloc[idx]`. 
            # Batch loop doesn't give us original indices easily unless we pass them.
            # Let's rely on retrieving from `test_df` via sample_id/model/mode combination later
            # Or better: Add them to dataset __getitem__?
            # Easier: Just zip with the dataframe since DataLoader is sequential and shuffle=False.
            pass

# Re-run loop with parallel zip of test_df to get metadata easily
eval_records = []

# Ensure test_df sort order matches loader
# (Dataset does reset_index, so simple iteration works)

idx_ptr = 0
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test Inference"):
        batch = {k: v.to(cfg.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            model_id=batch["model_id"],
            mode_id=batch["mode_id"],
        )
        
        utility_hat = outputs["utility_hat"]
        task_logits = outputs["task_logits"]
        task_preds = task_logits.argmax(dim=-1)
        
        # Collect results
        bs = len(batch["sample_ids"])
        current_df_slice = test_df.iloc[idx_ptr : idx_ptr + bs]
        idx_ptr += bs
        
        for i in range(bs):
            row = current_df_slice.iloc[i]
            rec = {
                "sample_id": row["sample_id"],
                "model_name": row["model_name"],
                "mode_name": row["mode_name"],
                "dataset": row.get("source_dataset", "unknown"),
                
                # Predictions
                "utility_pred": float(utility_hat[i].cpu()),
                "task_pred_id": int(task_preds[i].cpu()),
                "task_pred_name": task_names[int(task_preds[i].cpu())],
                
                # Targets / Truth
                "utility_target": float(batch["utility_target"][i].cpu()),
                "task_target_id": int(batch["task_id"][i].cpu()),
                "task_target_name": task_names[int(batch["task_id"][i].cpu())],

                # Metrics
                "actual_score": row["actual_score"],
                "actual_cost": row["actual_cost"],
                "actual_latency": row["actual_latency"],
            }
            eval_records.append(rec)

results_df = pd.DataFrame(eval_records)
print(f"Generated predictions for {len(results_df)} rows.")

Test Inference:   0%|          | 0/781 [00:00<?, ?it/s]

## 4. Analysis: Task Prediction

In [ ]:
print("Classification Report for Task Prediction:")
print(classification_report(results_df["task_target_name"], results_df["task_pred_name"]))

# Confusion Matrix
tasks = sorted(results_df["task_target_name"].unique())
cm = confusion_matrix(results_df["task_target_name"], results_df["task_pred_name"], labels=tasks)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, fmt='d', xticklabels=tasks, yticklabels=tasks, cmap="Blues")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Task Prediction Confusion Matrix')
plt.show()

## 5. Analysis: Router Performance (Best Model Selection)

For each query (sample_id, mode_id), we want to compare:
1. **Oracle**: The model with the highest `utility_target`.
2. **Router**: The model with the highest `utility_pred`.
3. **Random**: A random model.

Metrics to compute:
- **Router Accuracy**: % where Router Model == Oracle Model (or score is same).
- **Avg Score**: Average actual accuracy score of the selected model.
- **Avg Cost**: Average cost of selected model.
- **Avg Latency**: Average latency of selected model.

In [ ]:
def evaluate_routing(df):
    # Group by sample and mode
    groups = df.groupby(["sample_id", "mode_name"])
    
    records = []
    
    for (sid, mode), group in tqdm(groups, desc="Evaluating Routes"):
        # Oracle: Max utility target
        oracle_row = group.loc[group["utility_target"].idxmax()]
        
        # Router: Max utility pred
        router_row = group.loc[group["utility_pred"].idxmax()]
        
        # Random
        random_row = group.sample(1).iloc[0]
        
        records.append({
            "sample_id": sid,
            "mode_name": mode,
            "dataset": oracle_row["dataset"],
            
            # Selections
            "oracle_model": oracle_row["model_name"],
            "router_model": router_row["model_name"],
            
            # Scores
            "oracle_score": oracle_row["actual_score"],
            "router_score": router_row["actual_score"],
            "random_score": random_row["actual_score"],
            
            # Utility stats
            "router_confidence": router_row["utility_pred"],
            "router_utility_gap": oracle_row["utility_target"] - router_row["utility_target"], # How much worse was the choice?
            
            # Cost
            "oracle_cost": oracle_row["actual_cost"],
            "router_cost": router_row["actual_cost"],
            
            # Latency
            "oracle_lat": oracle_row["actual_latency"],
            "router_lat": router_row["actual_latency"],
        })
        
    return pd.DataFrame(records)

route_eval_df = evaluate_routing(results_df)
print(f"Evaluated routing for {len(route_eval_df)} queries.")

In [ ]:
# Aggregate Metrics per Mode
metrics = route_eval_df.groupby("mode_name").agg({
    "oracle_score": "mean",
    "router_score": "mean",
    "random_score": "mean",
    "oracle_cost": "mean",
    "router_cost": "mean",
    "oracle_lat": "mean",
    "router_lat": "mean"
}).reset_index()

metrics["accuracy_recovery"] = metrics["router_score"] / metrics["oracle_score"]
display(metrics)

## 6. Failure Analysis & Confidence Thresholds

When does the router fail? 
We define failure as `router_score < oracle_score` (significant drop).

We check:
1. **Failure Rate**: How often do we pick a suboptimal model?
2. **Confidence vs. Accuracy**: Are we less confident when we fail?
3. **Thresholding**: Can we fallback to a stronger model if confidence is low?

In [ ]:
# Define failure: Router picks a model with score < 95% of Oracle's score (allow small margin)
route_eval_df["is_failure"] = route_eval_df["router_score"] < (route_eval_df["oracle_score"] - 0.01)

print(f"Overall Failure Rate: {route_eval_df['is_failure'].mean():.2%}")

# Viz: Confidence distribution for Success vs Failure
plt.figure(figsize=(10, 6))
sns.histplot(data=route_eval_df, x="router_confidence", hue="is_failure", bins=30, kde=True, element="step")
plt.title("Confidence Distribution: Success vs Failure")
plt.xlabel("Predicted Utility (Confidence)")
plt.show()

In [ ]:
# Accuracy vs. Confidence Threshold
thresholds = np.linspace(0, 1.0, 20)
accuracies = []
coverage = []

for t in thresholds:
    # Subset where confidence >= t
    subset = route_eval_df[route_eval_df["router_confidence"] >= t]
    if len(subset) > 0:
        # Accuracy here means: How often did we NOT fail in this subset?
        acc = 1.0 - subset["is_failure"].mean()
        accuracies.append(acc)
        coverage.append(len(subset) / len(route_eval_df))
    else:
        accuracies.append(None)
        coverage.append(0)

plt.figure(figsize=(10, 6))
fig, ax1 = plt.subplots()

ax1.set_xlabel('Confidence Threshold')
ax1.set_ylabel('Router Reliability (Non-Failure Rate)', color='tab:blue')
ax1.plot(thresholds, accuracies, color='tab:blue', marker='o')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx() 
ax2.set_ylabel('Coverage (% of samples kept)', color='tab:orange')
ax2.plot(thresholds, coverage, color='tab:orange', linestyle='--')
ax2.tick_params(axis='y', labelcolor='tab:orange')

plt.title("Reliability vs Confidence Threshold")
plt.show()

## 7. CascadeFlow Comparison (Online)

We compare the Artemis Router (Predictive) against CascadeFlow (Sequential Confidence-based).
This requires loading the Ground Truth dataset since `router_profiles` lacks it.

In [ ]:
# Load Ground Truth Data (from router_test_trainer.parquet)
GT_DATA_PATH = PROJECT_ROOT / "dataset_old_copy" / "final_dataset" / "router_lexico" / "router_test_trainer.parquet"

if GT_DATA_PATH.exists():
    print(f"Loading Ground Truth from {GT_DATA_PATH}...")
    gt_df = pd.read_parquet(GT_DATA_PATH)
    
    # Filter to match the sample_ids in our test set
    test_sample_ids = set(test_df["sample_id"].unique())
    gt_metrics_df = gt_df[gt_df["sample_id"].isin(test_sample_ids)].copy()
    print(f"Loaded {len(gt_metrics_df)} samples with Ground Truth.")
else:
    print(f"WARNING: GT Data not found at {GT_DATA_PATH}. Comparison loop will be skipped.")
    gt_metrics_df = pd.DataFrame()

In [ ]:
# Cascade Configuration
MODEL_ENDPOINTS = {
    "Qwen/Qwen2.5-VL-3B-Instruct":   "http://localhost:8807/v1",
    "Qwen/Qwen2.5-VL-7B-Instruct":   "http://localhost:8804/v1",
    "google/gemma-3-27b-it":         "http://localhost:8800/v1",
    "Qwen/Qwen3-VL-8B-Thinking":     "http://localhost:8803/v1",
    "deepseek-ai/DeepSeek-OCR":      "http://localhost:8808/v1",
}

MODEL_PRICING = {
    "Qwen/Qwen2.5-VL-3B-Instruct":   {"prompt_per_1k": 0.0001, "completion_per_1k": 0.0001},
    "Qwen/Qwen2.5-VL-7B-Instruct":   {"prompt_per_1k": 0.0002, "completion_per_1k": 0.0002},
    "google/gemma-3-27b-it":         {"prompt_per_1k": 0.00009, "completion_per_1k": 0.00016},
    "Qwen/Qwen3-VL-8B-Thinking":     {"prompt_per_1k": 0.00018, "completion_per_1k": 0.0021},
    "deepseek-ai/DeepSeek-OCR":      {"prompt_per_1k": 0.00003, "completion_per_1k": 0.0001},
}

CASCADE_MODEL_ORDER = [
    "Qwen/Qwen2.5-VL-3B-Instruct",
    "Qwen/Qwen2.5-VL-7B-Instruct",
    "google/gemma-3-27b-it",
    "Qwen/Qwen3-VL-8B-Thinking",
    "deepseek-ai/DeepSeek-OCR",
]

QUALITY_THRESHOLD = 0.7  # For Cascade


In [ ]:
import asyncio
import time
import requests

async def run_cascade_eval(df, max_samples=None):
    if df.empty or CascadeAgent is None:
        return []
    
    # Check endpoints
    healthy = {}
    for name, url in MODEL_ENDPOINTS.items():
        try:
            requests.get(f"{url.rstrip('/v1')}/health", timeout=1) # Quick check
            healthy[name] = True
        except:
            # Assume up if connection refused? No, assume down.
            # Actually let's just try to assume they are up or use mock if flagged.
            # For simplicity in notebook, we assume user started them.
            healthy[name] = True

    configs = []
    for m in CASCADE_MODEL_ORDER:
        cfg = ModelConfig(
            name=m,
            provider="vllm",
            base_url=MODEL_ENDPOINTS[m],
            cost=MODEL_PRICING[m]["prompt_per_1k"],
            quality_threshold=QUALITY_THRESHOLD,
            metadata={"logical_name": m}
        )
        configs.append(cfg)
    
    agent = CascadeAgent(models=configs)
    
    results = []
    limit = max_samples if max_samples else len(df)
    
    print(f"Running Cascade on {limit} samples...")
    
    for i in range(limit):
        row = df.iloc[i]
        sample_id = row["sample_id"]
        prompt = row["prompt_raw"]
        gt = row["ground_truth"]
        gt_type = row.get("ground_truth_type", "exact")
        
        start_t = time.time()
        try:
            res = await agent.run(query=prompt, max_tokens=256)
            lat = (time.time() - start_t) * 1000
            
            pred = getattr(res, "content", "")
            model_used = res.model_used
            
            # Compute Cost
            meta = getattr(res, "metadata", {}) or {}
            p_tok = meta.get("prompt_tokens", 0)
            c_tok = meta.get("completion_tokens", 0)
            pricing = MODEL_PRICING.get(model_used, {"prompt_per_1k": 0, "completion_per_1k": 0})
            cost = (p_tok/1000)*pricing["prompt_per_1k"] + (c_tok/1000)*pricing["completion_per_1k"]
            
            # Score
            scores = Scorer.compute_all_scores(pred, gt, gt_type)
            is_correct = scores.get("is_correct", False)
            
            results.append({
                "sample_id": sample_id,
                "cascade_model": model_used,
                "cascade_cost": cost,
                "cascade_latency": lat,
                "cascade_correct": is_correct,
                "cascade_response": pred
            })
            
        except Exception as e:
            print(f"Error on {sample_id}: {e}")
            
        if i % 10 == 0: 
            print(f".", end="")
            
    return pd.DataFrame(results)

# Set MAX_SAMPLES to small number for testing, or None for full
cascade_results = await run_cascade_eval(gt_metrics_df, max_samples=100) 
if not cascade_results.empty:
    print(f"\nComputed Cascade Results: {len(cascade_results)}")
    display(cascade_results.head())

## 8. Comparative Analysis: Artemis vs Cascade

We merge the results on `sample_id` and compare.

In [ ]:
if not cascade_results.empty:
    # Get Artemis results for the same samples (using 'balanced' mode usually)
    # Filter route_eval_df for mode='balanced' (or whatever is default)
    artemis_subset = route_eval_df[route_eval_df["mode_name"] == "balanced"].copy()
    
    comparison = pd.merge(
        cascade_results, 
        artemis_subset[["sample_id", "router_model", "router_cost", "router_lat", "actual_score"]], 
        on="sample_id", 
        how="inner"
    )
    
    # Artemis Score is 'actual_score' (Molmo based, 0-10 or 0-100)
    # Cascade Score is 'cascade_correct' (Bool)
    # To compare, we might just look at trends or normalize.
    
    print(f"Comparing {len(comparison)} samples.")
    
    # Metrics
    avgs = comparison.agg({
        "cascade_cost": "mean",
        "router_cost": "mean",
        "cascade_latency": "mean",
        "router_lat": "mean",
        "cascade_correct": "mean",   # % Accuracy
        "actual_score": "mean"       # Avg Molmo Score
    })
    
    print("=== Result Summary ===")
    print(avgs)
    
    # Visualize specific comparison
    comparison["cost_diff"] = comparison["router_cost"] - comparison["cascade_cost"]
    comparison["lat_diff"] = comparison["router_lat"] - comparison["cascade_latency"]
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(comparison["cost_diff"], kde=True)
    plt.title("Cost Diff (Artemis - Cascade)")
    plt.axvline(0, color='r', linestyle='--')
    
    plt.subplot(1, 2, 2)
    sns.histplot(comparison["lat_diff"], kde=True)
    plt.title("Latency Diff (Artemis - Cascade)")
    plt.axvline(0, color='r', linestyle='--')
    plt.show()
else:
    print("Skipping comparison (No Cascade results).")